# Lesson 0 — NumPy for Embeddings: Vectors, Similarity & Semantic Search

**The hook:** in the next ~45 minutes you'll turn coffee drinks into **vectors** and write the exact math that powers "find me something similar" — the same **cosine similarity** that drives embedding search inside our *Matter Intelligence* capstone.

> **By the end you'll have shipped:** a working `most_similar()` recommender — the beating heart of semantic search over documents.
>
> This is **NumPy, Part 2.** Week 1 Day 4 gave you arrays, vectorization, masks, and aggregations. Here we add the real-world muscle — reshaping, NaN-aware stats, randomness, sorting, and **linear algebra** — and point it straight at LLM embeddings. Runs fully **offline, no API key** (pure NumPy).

|  |  |
|---|---|
| **Module** | M1 → M2 bridge · NumPy → Building with Claude |
| **Prerequisites** | Week 1 Day 4 — NumPy (Part 1) |
| **Est. time** | 40–50 minutes |
| **Capstone slice** | "Find matters similar to this one" — semantic search |
| **Difficulty** | Core (+ optional `Go Deeper 🔧`) |

*Runs end-to-end with **no API key and no internet** — it's pure NumPy. The vectors here are hand-made so you can see every number; in the Claude lessons they'll come from a real embedding model, but the math you write today is exactly the same.*

## What you'll be able to do

By the end of this lesson, you'll be able to:

1. Treat a row of numbers as a **vector**, and a table of them as a **matrix**.
2. **Reshape & combine** arrays — `reshape`, `newaxis`, `vstack`, `concatenate`.
3. Handle gaps with **NaN-aware** functions (`np.nan`, `nanmean`, `isnan`).
4. Generate **reproducible random** data with `np.random.default_rng`.
5. **Sort & rank** with `argsort` and grab a **top-k**.
6. Compute a **dot product** and a **vector norm**, then assemble **cosine similarity**.
7. Use it to **rank and recommend** — the engine of embedding search.

## Why it matters ⚖️

Every "smart" feature you're about to build on top of Claude — *find similar contracts, cluster matters, answer from our own documents* — rests on one idea: **turn text into a vector of numbers (an "embedding"), then measure which vectors point the same way.** That measurement is **cosine similarity**, and it's about ten lines of NumPy.

So this lesson is the hinge of the course. Part 1 taught you to *do math on a column*. Here you learn to *compare whole vectors* — and once you can rank vectors by similarity, you've built the core of semantic search, recommendations, and retrieval-augmented generation. We'll learn it on something you can eyeball (coffee drinks) and then swap in real embeddings later without changing the math.

## ⚙️ Setup — drinks as vectors

Run this first. We describe eight drinks by four **features** — `sweetness`, `caffeine`, `warmth`, `price` — so each drink becomes a 4-number **vector**, and the menu becomes a **matrix** (rows = drinks, columns = features). No files, no keys.

In [ ]:
import numpy as np

FEATURES = ["sweetness", "caffeine", "warmth", "price"]
DRINKS = {
    #                sweet caff warm price
    "Latte":        [3.0,  6.0, 9.0, 4.75],
    "Cappuccino":   [2.0,  6.0, 9.0, 4.50],
    "Espresso":     [1.0,  9.0, 8.0, 2.75],
    "Mocha":        [7.0,  5.0, 9.0, 5.25],
    "Cold Brew":    [2.0,  8.0, 1.0, 4.65],
    "Iced Latte":   [3.0,  6.0, 2.0, 4.75],
    "Hot Chocolate":[8.0,  1.0, 9.0, 4.00],
    "Drip":         [1.0,  7.0, 8.0, 2.50],
}

names = list(DRINKS)
matrix = np.array(list(DRINKS.values()))   # shape (8 drinks, 4 features)

print(f"numpy {np.__version__}")
print(f"names: {names}")
print(f"matrix shape (drinks, features): {matrix.shape}")
print(matrix)

## 1 · A drink is a vector; the menu is a matrix

`matrix` is 2-D: **8 rows** (drinks) × **4 columns** (features). Row `i` is one drink's vector; column `j` is one feature across all drinks. This row/column duality is the whole game — an embedding table looks *exactly* like this, just with hundreds of columns instead of four.

In [ ]:
latte = matrix[0]          # first row  -> Latte's vector
print(f"Latte vector: {latte}   (shape {latte.shape})")

caffeine_col = matrix[:, 1]  # every row, column 1 -> caffeine across all drinks
print(f"caffeine column: {caffeine_col}")

# index a single cell: Mocha's sweetness (row 3, col 0)
print(f"Mocha sweetness: {matrix[3, 0]}")

**What just happened:** `matrix[0]` pulled a **row** (one drink), `matrix[:, 1]` pulled a **column** (one feature everywhere), and `matrix[3, 0]` pulled one cell. Two-axis indexing `[row, col]` is how you move around any 2-D array.

## 2 · Reshaping & shape manipulation

You'll constantly need to change an array's shape — flatten it, stand a row up as a column, or add an axis so shapes line up for math. `reshape`, `ravel`, `.T`, and `np.newaxis` are the tools.

In [ ]:
print(f"ravel (flatten to 1-D): {matrix.ravel()[:8]} ...")     # all 32 numbers in a row
print(f"transpose .T shape: {matrix.T.shape}")                 # (4 features, 8 drinks)

# newaxis turns a 1-D vector (4,) into a column (4,1) or row (1,4) — crucial for broadcasting
print(f"latte as column shape: {latte[:, np.newaxis].shape}")
print(f"reshape 8x4 -> 4x8: {matrix.reshape(4, 8).shape}")     # same 32 numbers, new grid

> **`Common gotcha ⚠️`** — `reshape` only works if the total number of elements matches (8×4 = 4×8 = 32). Use `-1` to let NumPy infer one dimension: `matrix.reshape(-1, 2)` → `(16, 2)`.

## 3 · Combining arrays — add a drink

New data arrives all the time. `np.vstack` stacks rows (add a drink); `np.concatenate` is the general tool; `np.hstack` glues columns (add a feature).

In [ ]:
affogato = np.array([5.0, 8.0, 6.0, 5.50])          # a new drink vector
menu2 = np.vstack([matrix, affogato])               # add it as a new row
print(f"was {matrix.shape[0]} drinks, now {menu2.shape[0]}")

# add a feature column (e.g. a 'foam' score) with hstack
foam = np.array([[8], [9], [1], [7], [0], [2], [9], [1]])
with_foam = np.hstack([matrix, foam])
print(f"added a feature: {matrix.shape} -> {with_foam.shape}")

## 4 · NaN-aware stats — real data has holes

Real feature tables have missing values. NumPy marks them `np.nan`. The catch: **any normal stat touching a NaN returns NaN**. The fix: the `nan*` family (`nanmean`, `nansum`, …) skips them, and `np.isnan` finds them.

In [ ]:
prices = matrix[:, 3].copy()
prices[4] = np.nan                     # pretend Cold Brew's price went missing

print(f"plain mean (poisoned by NaN): {prices.mean()}")
print(f"nanmean (skips the gap):      {round(np.nanmean(prices), 2)}")
print(f"where are the NaNs? {np.where(np.isnan(prices))[0]}")

# a common cleanup: fill gaps with the column's NaN-safe mean
prices[np.isnan(prices)] = np.nanmean(prices)
print(f"after fill: {np.round(prices, 2)}")

## 5 · Randomness & sampling — reproducibly

Simulations, test data, shuffling, sampling a subset — all need random numbers. Modern NumPy uses a **`Generator`** from `default_rng(seed)`. **Seed it** and results repeat exactly (essential for reproducible experiments).

In [ ]:
rng = np.random.default_rng(42)        # seeded -> same numbers every run

print(f"3 random drink indices: {rng.integers(0, len(names), size=3)}")
print(f"a random 2x4 feature block:\n{np.round(rng.normal(5, 2, size=(2, 4)), 2)}")

# sample 3 drinks without replacement, and shuffle a deck of indices
print(f"sample 3 drinks: {rng.choice(names, size=3, replace=False)}")
deck = np.arange(len(names)); rng.shuffle(deck)
print(f"shuffled order: {deck}")

## 6 · Sorting & searching — rank and take top-k

To recommend, you rank. `np.sort` sorts values; **`np.argsort` returns the *indices*** that would sort them — which is how you reorder the *names* to match. A top-k is just a slice of the argsort.

In [ ]:
price_col = matrix[:, 3]
order = np.argsort(price_col)          # indices from cheapest to priciest
print(f"cheapest -> priciest: {[names[i] for i in order]}")

# top-3 most caffeinated: argsort descending, take first 3
caff = matrix[:, 1]
top3 = np.argsort(caff)[::-1][:3]
for rank, i in enumerate(top3, 1):
    print(f"  #{rank} {names[i]:<13} caffeine={caff[i]}")

## 7 · The two operations behind similarity: **dot** & **norm**

Comparing vectors comes down to two pieces:

- **Dot product** (`a @ b` or `np.dot`) — multiply element-wise and sum. It's large when two vectors are big *and* pointing the same way.
- **Norm** (`np.linalg.norm`) — a vector's length, `sqrt(sum(v**2))`. It's how we cancel out "bigger numbers" so we compare *direction*, not *magnitude*.

In [ ]:
a = matrix[0]   # Latte
b = matrix[1]   # Cappuccino
c = matrix[4]   # Cold Brew

print(f"Latte · Cappuccino = {a @ b}")          # @ is the dot-product operator
print(f"Latte · Cold Brew  = {a @ c}")
print(f"|Latte| (norm) = {round(np.linalg.norm(a), 2)}")
print(f"|Cold Brew|    = {round(np.linalg.norm(c), 2)}")

**What just happened:** the dot product mixes magnitude *and* direction — that's why we divide by the norms next, to isolate direction. Two vectors pointing the same way score ~1 no matter how long they are.

## 8 · Cosine similarity — the payoff

**Cosine similarity** = dot product ÷ (norm × norm). It ranges from **1** (identical direction) to **0** (unrelated) to **−1** (opposite). It's *the* standard way to compare embeddings because it ignores length and measures only *alignment*.

In [ ]:
def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"Latte vs Cappuccino: {cosine(matrix[0], matrix[1]):.3f}")   # very alike
print(f"Latte vs Cold Brew:  {cosine(matrix[0], matrix[4]):.3f}")   # warm vs iced
print(f"Espresso vs Drip:    {cosine(matrix[2], matrix[7]):.3f}")   # both strong & hot

### One query against *everything*, vectorized

You rarely compare two vectors — you compare **one query against a whole matrix** and rank. Instead of looping, do it in one shot: normalize every row, then a single matrix-vector product gives all similarities at once.

In [ ]:
def cosine_to_all(query, mat):
    # normalize each row to unit length, and the query too
    mat_unit = mat / np.linalg.norm(mat, axis=1, keepdims=True)   # axis=1 + keepdims -> (n,1) broadcast
    q_unit = query / np.linalg.norm(query)
    return mat_unit @ q_unit                                      # (n,4) @ (4,) -> (n,) similarities

sims = cosine_to_all(matrix[0], matrix)      # Latte vs all drinks
for i in np.argsort(sims)[::-1]:             # rank most -> least similar
    print(f"  {names[i]:<13} {sims[i]:.3f}")

> **`Go Deeper 🔧` — normalize once, then it's just a dot.** If you unit-normalize every embedding up front, cosine similarity *is* the dot product — so a full **similarity matrix** for N items is one `M @ M.T` (matmul). That single line is what a vector database does millions of times a second.

In [ ]:
unit = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
sim_matrix = unit @ unit.T                   # (8,8) every drink vs every drink
print("similarity matrix shape:", sim_matrix.shape)
print(np.round(sim_matrix, 2))

> **`Common pitfalls ⚠️`**
>
> - **Feature scale skews raw vectors** — here `price` (~2–5) and `caffeine` (~1–9) differ in range, so before similarity on *real* mixed features you'd standardize columns (subtract `mean`, divide `std`, `axis=0`). Embeddings come pre-scaled, so this bites mostly with hand-made features.
> - **Shapes must align for `@`** — `(n,4) @ (4,)` works; `(n,4) @ (n,4)` doesn't. Check `.shape` when matmul errors.
> - **NaN poisons a vector** — one `np.nan` makes the whole similarity `NaN`. Clean/impute first (§4).
> - **`axis` + `keepdims=True`** keeps the `(n,1)` shape so it broadcasts cleanly when dividing rows by their norms.

## ✍️ Your turn

In [ ]:
# Using `matrix`, `names`, and cosine_to_all():
# TODO 1: compute similarities of 'Mocha' (index 3) against all drinks
# TODO 2: print the drinks ranked most -> least similar to Mocha
# TODO 3: what's the single most similar drink to Mocha (excluding itself)?
# TODO 4 (stretch): standardize the columns first (subtract mean, divide std, axis=0),
#         then recompute — does Mocha's nearest neighbour change?

# your code here


<details><summary>✅ Show solution</summary>

```python
sims = cosine_to_all(matrix[3], matrix)
order = np.argsort(sims)[::-1]
for i in order:
    print(f"{names[i]:<13} {sims[i]:.3f}")

best = order[order != 3][0]          # skip Mocha itself
print("nearest:", names[best])

# 4 — standardized
z = (matrix - matrix.mean(axis=0)) / matrix.std(axis=0)
sims_z = cosine_to_all(z[3], z)
print("nearest (standardized):", names[np.argsort(sims_z)[::-1][1]])
```
</details>

## 🚀 Build the artifact — a `most_similar()` recommender

Package it into one reusable function: given a drink name, return its top-k nearest neighbours by cosine similarity. This is a *recommendation engine* — and structurally identical to searching documents by meaning.

In [ ]:
def most_similar(name, mat, names, k=3):
    """Return the k drinks most similar to `name` (excluding itself)."""
    idx = names.index(name)
    sims = cosine_to_all(mat[idx], mat)
    ranked = np.argsort(sims)[::-1]
    hits = [i for i in ranked if i != idx][:k]
    return [(names[i], round(float(sims[i]), 3)) for i in hits]

for drink in ["Mocha", "Cold Brew", "Espresso"]:
    print(f"Because you like {drink}, try:")
    for rec, score in most_similar(drink, matrix, names, k=2):
        print(f"    {rec:<13} (similarity {score})")
    print()

print("✅ Shipped: most_similar() — a cosine-similarity recommender.")

> **🔗 Your world — from drinks to matters (and to Claude).** Swap the four hand-made features for a real **embedding vector** — a few hundred numbers a model produces from a matter's text — and `most_similar()` becomes **semantic search over your matters**: *"find prior matters like this new dispute."* Nothing about the math changes; only the vectors get richer. In the very next lesson we call **Claude** to turn text into those vectors — this is the NumPy engine that will rank them.

## 📝 Recap — what you shipped

- A **vector** is a row of numbers; a **matrix** is a stack of them (rows × features).
- **`reshape` / `newaxis` / `vstack` / `concatenate`** change and combine shapes.
- **NaN-aware** functions (`nanmean`, `isnan`) survive missing data.
- **`default_rng(seed)`** gives reproducible randomness; **`argsort`** ranks and takes top-k.
- **Cosine similarity** = `dot / (norm·norm)` — vectorized as `unit @ unit.T`.
- **Artifact:** `most_similar()`, a recommender that *is* embedding search.

## 🧠 Check your understanding

1. Why divide the dot product by the norms — what does that remove?
2. What does `np.argsort(sims)[::-1]` give you, and why the `[::-1]`?
3. If a query vector contains one `np.nan`, what does its cosine similarity come out as, and how do you prevent it?
4. In `np.linalg.norm(mat, axis=1, keepdims=True)`, what is `axis=1` doing?

<details><summary>Answers</summary>

1. It removes **magnitude**, leaving only **direction** — so long and short vectors that point the same way score alike (that's what makes it *cosine* similarity).
2. The indices that sort `sims` **ascending**, reversed to **descending** — i.e. most-similar first.
3. `NaN` — any arithmetic with `NaN` propagates. Clean or impute the vector first (e.g. `nanmean` fill).
4. Computing the norm **across the features of each row** (one length per drink), and `keepdims=True` keeps it shaped `(n,1)` so it broadcasts when dividing each row.
</details>

## ➡️ Next up — Lesson 1: your first call to Claude

You can now compare and rank vectors — the machinery under embedding search. Next, **Lesson 1** hands the other half of the puzzle: calling **Claude** to turn a contract clause into plain English (and, soon, into the embedding vectors you just learned to rank). NumPy ranks; Claude understands — together they're *Matter Intelligence*.

## 📖 Reference & glossary

| Term | Plain meaning | Why it matters here |
|---|---|---|
| vector | a 1-D array of numbers | one drink / one embedding |
| matrix | a 2-D array (rows × cols) | the menu / the embedding table |
| `reshape` / `newaxis` | change shape / add an axis | line shapes up for math |
| `nanmean` / `isnan` | NaN-skipping stat / find gaps | survive missing features |
| `default_rng(seed)` | reproducible random generator | sampling, test data |
| `argsort` | indices that would sort | ranking, top-k |
| dot product (`@`) | element-wise multiply & sum | raw similarity signal |
| `np.linalg.norm` | vector length | cancels magnitude |
| cosine similarity | `dot / (norm·norm)` | the standard embedding compare |

**Official docs:** [Linear algebra (`numpy.linalg`)](https://numpy.org/doc/stable/reference/routines.linalg.html) · [Random `Generator`](https://numpy.org/doc/stable/reference/random/generator.html) · [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)